# MetaMIRAGE Concurrent Preload — Wave Finalizer

This notebook is the **only preload component allowed to perform wave-level global finalization**.

It runs **after every state in `EXPECTED_STATES` has completed**.

Responsibilities:

1. Validate every expected state manifest.
2. Validate coordinator `COMPLETE` status for every expected state.
3. Verify manifest hashes match coordinator records.
4. Verify all states in the wave used the same global crop seed.
5. Merge each state's `crop_occurrences_state.json` into the maintained global `crop_occurrences.json`.
6. Back up and atomically replace the global crop JSON.
7. Validate the shared Qdrant collection.
8. Create a cumulative Qdrant snapshot.
9. Optionally download the snapshot into shared persistent storage.
10. Write the final wave manifest **last** as the commit marker.

## Important

- Wave size is variable.
- `EXPECTED_STATES` is edited manually for each wave.
- State workers must not be running/writing when this notebook snapshots the wave.
- This notebook never deletes/resets/restores Qdrant.
- `RUN_FINALIZATION=False` by default so validation can be run safely before global mutation.


## 1. Configuration

Edit this block for each wave.

Expected repository shape:

```text
preload_pipeline/
├── workers/
├── persistent_state/
│   └── <BUILD_ID>/
│       ├── IL/
│       │   ├── state_manifest.json
│       │   └── crop_occurrences_state.json
│       └── ...
├── shared/
│   ├── crop_occurrences.json
│   ├── finalization/
│   └── snapshots/
└── finalizer/
    └── finalize_wave.ipynb
```


In [ ]:
from pathlib import Path
import os
import re

# ------------------------------------------------------------
# EDIT FOR EACH WAVE
# ------------------------------------------------------------

BUILD_ID = "build_2026_08"
WAVE_ID = "wave_01"

# Variable size: list exactly the states that belong to this wave.
EXPECTED_STATES = [
    "IL",
    "IA",
    "IN",
]

# Service endpoints.
COORDINATOR_URL = os.environ.get(
    "METAMIRAGE_COORDINATOR_URL",
    "http://127.0.0.1:8001",
).rstrip("/")

QDRANT_URL = os.environ.get(
    "QDRANT_URL",
    "http://127.0.0.1:6333",
).rstrip("/")

QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY") or None
QDRANT_COLLECTION = "mirage_base_build"

# Snapshot policy.
DOWNLOAD_SNAPSHOT = True

# Safety switch.
# Leave False while running validation/preview cells.
# Change to True only when you are ready to merge crop JSON + snapshot Qdrant.
RUN_FINALIZATION = False

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

FINALIZER_DIR = Path.cwd().resolve()
PIPELINE_ROOT = (
    FINALIZER_DIR.parent
    if FINALIZER_DIR.name.lower() == "finalizer"
    else FINALIZER_DIR
).resolve()

PERSISTENT_STATE_ROOT = PIPELINE_ROOT / "persistent_state"
SHARED_DIR = PIPELINE_ROOT / "shared"

GLOBAL_CROP_JSON = SHARED_DIR / "crop_occurrences.json"

FINALIZATION_ROOT = SHARED_DIR / "finalization" / BUILD_ID / WAVE_ID
FINALIZATION_JOURNAL = FINALIZATION_ROOT / "finalization_state.json"
WAVE_MANIFEST_PATH = FINALIZATION_ROOT / "wave_manifest.json"

CROP_BACKUP_DIR = SHARED_DIR / "crop_backups" / BUILD_ID
SNAPSHOT_OUTPUT_DIR = SHARED_DIR / "snapshots" / BUILD_ID / WAVE_ID

for path in [
    SHARED_DIR,
    FINALIZATION_ROOT,
    CROP_BACKUP_DIR,
    SNAPSHOT_OUTPUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

BUILD_ID = BUILD_ID.strip()
WAVE_ID = WAVE_ID.strip()
EXPECTED_STATES = [str(s).strip().upper() for s in EXPECTED_STATES]

if not BUILD_ID:
    raise ValueError("BUILD_ID must not be empty.")
if not WAVE_ID:
    raise ValueError("WAVE_ID must not be empty.")
if not EXPECTED_STATES:
    raise ValueError("EXPECTED_STATES must contain at least one state.")
if len(EXPECTED_STATES) != len(set(EXPECTED_STATES)):
    raise ValueError("EXPECTED_STATES contains duplicate state codes.")
for state in EXPECTED_STATES:
    if not re.fullmatch(r"[A-Z]{2}", state):
        raise ValueError(f"Invalid state code: {state!r}")

print("Pipeline root:", PIPELINE_ROOT)
print("Build:", BUILD_ID)
print("Wave:", WAVE_ID)
print("Expected states:", EXPECTED_STATES)
print("Coordinator:", COORDINATOR_URL)
print("Qdrant:", QDRANT_URL)
print("Collection:", QDRANT_COLLECTION)
print("Global crop JSON:", GLOBAL_CROP_JSON)
print("Finalization journal:", FINALIZATION_JOURNAL)
print("Wave manifest:", WAVE_MANIFEST_PATH)
print("RUN_FINALIZATION:", RUN_FINALIZATION)


## 2. Imports and utility helpers

In [ ]:
import hashlib
import json
import socket
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import requests


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def load_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def write_json_atomic(path: Path, payload: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    os.replace(tmp, path)


def qdrant_headers() -> Dict[str, str]:
    return {"api-key": QDRANT_API_KEY} if QDRANT_API_KEY else {}


def coordinator_get(path: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    r = requests.get(
        f"{COORDINATOR_URL}{path}",
        params=params,
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    if not isinstance(data, dict):
        raise RuntimeError(f"Coordinator returned non-object JSON: {data!r}")
    return data


def qdrant_get(path: str, *, stream: bool = False, timeout=60):
    r = requests.get(
        f"{QDRANT_URL}{path}",
        headers=qdrant_headers(),
        stream=stream,
        timeout=timeout,
    )
    r.raise_for_status()
    return r


def qdrant_post(path: str, payload: Optional[Dict[str, Any]] = None, timeout=120):
    r = requests.post(
        f"{QDRANT_URL}{path}",
        headers=qdrant_headers(),
        json=payload,
        timeout=timeout,
    )
    r.raise_for_status()
    return r


def normalize_name(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "").strip().lower())


print("Utilities ready.")


## 3. Service connectivity preflight

This is read-only.


In [ ]:
def service_preflight() -> Dict[str, Any]:
    # Coordinator
    coordinator_health = coordinator_get("/health")

    # Qdrant
    qdrant_response = qdrant_get("/collections", timeout=30)
    qdrant_collections = qdrant_response.json()

    collection_names = {
        item.get("name")
        for item in (
            qdrant_collections.get("result", {}).get("collections", [])
            if isinstance(qdrant_collections, dict)
            else []
        )
    }

    if QDRANT_COLLECTION not in collection_names:
        raise RuntimeError(
            f"Required Qdrant collection {QDRANT_COLLECTION!r} does not exist. "
            f"Available: {sorted(x for x in collection_names if x)}"
        )

    print("✅ Coordinator reachable")
    print("✅ Qdrant reachable")
    print("✅ Collection exists:", QDRANT_COLLECTION)

    return {
        "coordinator_health": coordinator_health,
        "qdrant_collections": qdrant_collections,
    }


SERVICE_PREFLIGHT = service_preflight()


## 4. Load and validate every expected state

The finalizer requires **both**:

- a complete state manifest on shared persistent storage; and
- coordinator status `complete` for that same state.

The coordinator's recorded manifest SHA256 must match the actual state manifest file.


In [ ]:
def state_dir(state_code: str) -> Path:
    return PERSISTENT_STATE_ROOT / BUILD_ID / state_code


def state_manifest_path(state_code: str) -> Path:
    return state_dir(state_code) / "state_manifest.json"


def state_crop_path(state_code: str) -> Path:
    return state_dir(state_code) / "crop_occurrences_state.json"


def load_and_validate_state(state_code: str) -> Dict[str, Any]:
    manifest_path = state_manifest_path(state_code)
    crop_path = state_crop_path(state_code)

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"State {state_code}: manifest missing: {manifest_path}"
        )
    if not crop_path.exists():
        raise FileNotFoundError(
            f"State {state_code}: crop artifact missing: {crop_path}"
        )

    manifest = load_json(manifest_path)
    if not isinstance(manifest, dict):
        raise ValueError(f"State {state_code}: invalid manifest JSON.")

    if manifest.get("artifact_type") != "concurrent_preload_state_manifest":
        raise ValueError(
            f"State {state_code}: unexpected artifact_type "
            f"{manifest.get('artifact_type')!r}"
        )
    if manifest.get("status") != "complete":
        raise ValueError(
            f"State {state_code}: manifest status is not complete."
        )
    if manifest.get("build_id") != BUILD_ID:
        raise ValueError(
            f"State {state_code}: manifest BUILD_ID mismatch: "
            f"{manifest.get('build_id')!r} != {BUILD_ID!r}"
        )
    if manifest.get("wave_id") != WAVE_ID:
        raise ValueError(
            f"State {state_code}: manifest WAVE_ID mismatch: "
            f"{manifest.get('wave_id')!r} != {WAVE_ID!r}"
        )

    processed = manifest.get("state_processed") or {}
    if str(processed.get("code") or "").upper() != state_code:
        raise ValueError(
            f"State {state_code}: manifest state code mismatch."
        )

    services = manifest.get("services") or {}
    if services.get("qdrant_collection") != QDRANT_COLLECTION:
        raise ValueError(
            f"State {state_code}: Qdrant collection mismatch: "
            f"{services.get('qdrant_collection')!r}"
        )

    # Verify crop artifact hash against state manifest.
    crop_info = manifest.get("crop_dictionary") or {}
    expected_crop_sha = crop_info.get("sha256")
    actual_crop_sha = file_sha256(crop_path)

    if expected_crop_sha and expected_crop_sha != actual_crop_sha:
        raise RuntimeError(
            f"State {state_code}: crop artifact hash mismatch."
        )

    # Validate crop artifact identity.
    crop_payload = load_json(crop_path)
    if not isinstance(crop_payload, dict):
        raise ValueError(f"State {state_code}: invalid crop artifact.")
    if crop_payload.get("build_id") != BUILD_ID:
        raise ValueError(f"State {state_code}: crop BUILD_ID mismatch.")
    if crop_payload.get("wave_id") != WAVE_ID:
        raise ValueError(f"State {state_code}: crop WAVE_ID mismatch.")
    if str(crop_payload.get("state_code") or "").upper() != state_code:
        raise ValueError(f"State {state_code}: crop state-code mismatch.")
    if not isinstance(crop_payload.get("crop_occurrences"), dict):
        raise ValueError(
            f"State {state_code}: crop_occurrences must be an object."
        )

    # Coordinator must agree.
    coordinator = coordinator_get(
        "/state/status",
        {
            "build_id": BUILD_ID,
            "state_code": state_code,
        },
    )

    if str(coordinator.get("status") or "").lower() != "complete":
        raise RuntimeError(
            f"State {state_code}: coordinator status is not COMPLETE: "
            f"{coordinator}"
        )
    if coordinator.get("wave_id") != WAVE_ID:
        raise RuntimeError(
            f"State {state_code}: coordinator wave mismatch."
        )

    manifest_sha = file_sha256(manifest_path)
    coordinator_manifest_sha = coordinator.get("manifest_sha256")
    if coordinator_manifest_sha != manifest_sha:
        raise RuntimeError(
            f"State {state_code}: manifest SHA mismatch between "
            "shared storage and coordinator."
        )

    return {
        "state_code": state_code,
        "manifest_path": manifest_path,
        "manifest_sha256": manifest_sha,
        "manifest": manifest,
        "crop_path": crop_path,
        "crop_sha256": actual_crop_sha,
        "crop_payload": crop_payload,
        "coordinator": coordinator,
    }


STATE_RESULTS = {
    state: load_and_validate_state(state)
    for state in EXPECTED_STATES
}

print("✅ All expected states have valid manifests + coordinator COMPLETE status.")
for state, result in STATE_RESULTS.items():
    print(
        state,
        "manifest=", result["manifest_path"],
        "crop=", result["crop_path"],
    )


## 5. Cross-state contract validation

States in one wave must agree on important build contracts.

The finalizer also checks that every worker started from the same global crop seed.


In [ ]:
def validate_cross_state_contracts(state_results: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    manifests = [item["manifest"] for item in state_results.values()]

    # Same pipeline contract across the wave.
    contract_fields = {
        "pipeline_version": set(),
        "embedding_model": set(),
        "metadata_contract": set(),
        "chunk_logical_id": set(),
        "qdrant_collection": set(),
    }

    crop_seed_hashes = set()

    for manifest in manifests:
        versions = manifest.get("versions") or {}
        services = manifest.get("services") or {}
        chunk_identity = manifest.get("chunk_identity") or {}
        fingerprints = manifest.get("input_fingerprints") or {}
        global_seed = fingerprints.get("global_crop_seed") or {}

        contract_fields["pipeline_version"].add(manifest.get("pipeline_version"))
        contract_fields["embedding_model"].add(versions.get("embedding_model"))
        contract_fields["metadata_contract"].add(versions.get("metadata_contract"))
        contract_fields["chunk_logical_id"].add(chunk_identity.get("logical_id"))
        contract_fields["qdrant_collection"].add(services.get("qdrant_collection"))

        seed_sha = global_seed.get("sha256")
        if seed_sha:
            crop_seed_hashes.add(seed_sha)

    bad = {
        key: sorted(repr(v) for v in values)
        for key, values in contract_fields.items()
        if len(values) != 1
    }
    if bad:
        raise RuntimeError(
            "States in this wave disagree on build contracts:\n"
            + json.dumps(bad, indent=2)
        )

    if len(crop_seed_hashes) != 1:
        raise RuntimeError(
            "Expected every state in the wave to have the same global crop seed hash, "
            f"but found: {sorted(crop_seed_hashes)}"
        )

    crop_seed_sha = next(iter(crop_seed_hashes))

    result = {
        "contracts": {
            key: next(iter(values))
            for key, values in contract_fields.items()
        },
        "crop_seed_sha256": crop_seed_sha,
    }

    print("✅ Cross-state contracts consistent.")
    print(json.dumps(result, indent=2))
    return result


CROSS_STATE_CONTRACTS = validate_cross_state_contracts(STATE_RESULTS)


## 6. Preview cumulative crop merge

This cell is read-only.

It shows which state keys will be added/replaced in the global maintained crop JSON.


In [ ]:
if not GLOBAL_CROP_JSON.exists():
    raise FileNotFoundError(
        f"Global crop JSON does not exist: {GLOBAL_CROP_JSON}"
    )

CURRENT_GLOBAL_CROP = load_json(GLOBAL_CROP_JSON)
if not isinstance(CURRENT_GLOBAL_CROP, dict):
    raise ValueError("Global crop_occurrences.json must be a top-level object.")

CURRENT_GLOBAL_CROP_SHA = file_sha256(GLOBAL_CROP_JSON)
WAVE_SEED_SHA = CROSS_STATE_CONTRACTS["crop_seed_sha256"]

journal = load_json(FINALIZATION_JOURNAL, {}) or {}
journal_crop_after_sha = (
    (journal.get("crop_merge") or {}).get("after_sha256")
    if isinstance(journal, dict)
    else None
)

if CURRENT_GLOBAL_CROP_SHA == WAVE_SEED_SHA:
    CROP_MERGE_STATE = "seed_ready"
elif journal_crop_after_sha and CURRENT_GLOBAL_CROP_SHA == journal_crop_after_sha:
    CROP_MERGE_STATE = "already_merged_by_this_finalizer"
else:
    # Recovery case: journal may have been written before merge and the kernel
    # may have died after os.replace but before journal update. Accept only if
    # each expected state's current section exactly matches its state artifact.
    sections_match = True
    for state, result in STATE_RESULTS.items():
        payload = result["crop_payload"]
        state_key = normalize_name(payload["state_key"])
        expected_section = payload["crop_occurrences"]
        if CURRENT_GLOBAL_CROP.get(state_key) != expected_section:
            sections_match = False
            break

    if (
        isinstance(journal, dict)
        and journal.get("status") in {"started", "crop_merging", "crop_merged", "snapshotting"}
        and sections_match
    ):
        CROP_MERGE_STATE = "recovered_already_merged"
    else:
        raise RuntimeError(
            "Current global crop JSON does not match this wave's seed hash and cannot "
            "be safely identified as this wave's already-applied merge.\n"
            f"Current: {CURRENT_GLOBAL_CROP_SHA}\n"
            f"Wave seed: {WAVE_SEED_SHA}\n"
            "Do not overwrite it until the wave/finalization ordering is reconciled."
        )

print("Current global crop SHA:", CURRENT_GLOBAL_CROP_SHA)
print("Wave seed SHA:", WAVE_SEED_SHA)
print("Crop merge state:", CROP_MERGE_STATE)

print("\nWave crop sections:")
for state, result in STATE_RESULTS.items():
    crop = result["crop_payload"]
    state_key = normalize_name(crop["state_key"])
    crop_count = len(crop["crop_occurrences"])
    action = "replace" if state_key in CURRENT_GLOBAL_CROP else "add"
    print(f"  {state}: key={state_key!r}, crops={crop_count}, action={action}")


## 7. Qdrant structural validation

Read-only checks:

- collection exists;
- collection info is readable;
- exact point count is readable;
- a small payload sample can be scrolled.

No embedding model is required.


In [ ]:
REQUIRED_SAMPLE_PAYLOAD_KEYS = {
    "source_type",
    "source_id",
    "title",
    "page",
    "chunk_index",
    "content_hash",
}


def qdrant_collection_info() -> Dict[str, Any]:
    r = qdrant_get(
        f"/collections/{QDRANT_COLLECTION}",
        timeout=60,
    )
    data = r.json()
    if not isinstance(data, dict) or data.get("status") != "ok":
        raise RuntimeError(f"Unexpected Qdrant collection response: {data}")
    return data


def qdrant_exact_count() -> int:
    r = qdrant_post(
        f"/collections/{QDRANT_COLLECTION}/points/count",
        {"exact": True},
        timeout=120,
    )
    data = r.json()
    return int((data.get("result") or {}).get("count", 0))


def qdrant_payload_sample(limit: int = 20) -> List[Dict[str, Any]]:
    r = qdrant_post(
        f"/collections/{QDRANT_COLLECTION}/points/scroll",
        {
            "limit": limit,
            "with_payload": True,
            "with_vector": False,
        },
        timeout=120,
    )
    data = r.json()
    points = (data.get("result") or {}).get("points", [])
    return points if isinstance(points, list) else []


QDRANT_INFO = qdrant_collection_info()
QDRANT_POINT_COUNT = qdrant_exact_count()
QDRANT_SAMPLE = qdrant_payload_sample()

if QDRANT_POINT_COUNT <= 0:
    raise RuntimeError("Qdrant collection has zero points; refusing wave finalization.")

sample_contract_issues = []
for point in QDRANT_SAMPLE:
    payload = point.get("payload") or {}
    missing = sorted(k for k in REQUIRED_SAMPLE_PAYLOAD_KEYS if k not in payload)
    if missing:
        sample_contract_issues.append(
            {"point_id": point.get("id"), "missing": missing}
        )

if sample_contract_issues:
    print("⚠️ Payload sample contract warnings:")
    print(json.dumps(sample_contract_issues[:10], indent=2))
else:
    print("✅ Payload sample contains required canonical keys.")

print("✅ Qdrant collection readable.")
print("Qdrant exact point count:", QDRANT_POINT_COUNT)
print("Sample points inspected:", len(QDRANT_SAMPLE))


## 8. Aggregate wave statistics

Read-only aggregation from state manifests.


In [ ]:
def aggregate_wave_statistics(state_results: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    numeric_keys = [
        "sources_discovered",
        "duplicates_skipped",
        "sources_permanently_failed",
        "documents_accepted",
        "documents_rejected",
        "documents_permanently_failed",
        "qualification_chunks_processed",
        "qualification_chunks_permanently_failed",
        "rag_chunks_created",
        "rag_chunks_indexed",
        "rag_chunks_duplicate_skipped",
        "rag_chunks_permanently_failed",
    ]

    totals = {key: 0 for key in numeric_keys}
    per_state = {}

    for state, result in state_results.items():
        stats = (result["manifest"].get("state_statistics") or {})
        per_state[state] = stats
        for key in numeric_keys:
            totals[key] += int(stats.get(key, 0) or 0)

    return {
        "wave_totals": totals,
        "per_state": per_state,
    }


WAVE_STATS = aggregate_wave_statistics(STATE_RESULTS)
print(json.dumps(WAVE_STATS["wave_totals"], indent=2))


## 9. Finalization implementation

The actual mutating flow is:

```text
validate again
→ acquire manual finalizer lock
→ initialize/recover journal
→ merge global crop JSON atomically
→ validate Qdrant again
→ create cumulative Qdrant snapshot
→ optionally download snapshot
→ write wave manifest LAST
```

The finalizer uses a shared lock file to prevent accidentally running two finalizers at once.

If a stale lock remains after a crashed finalizer, inspect the metadata before removing it manually.


In [ ]:
FINALIZER_LOCK_PATH = SHARED_DIR / ".finalize_wave.lock"


def acquire_finalizer_lock():
    metadata = {
        "build_id": BUILD_ID,
        "wave_id": WAVE_ID,
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
        "acquired_at": utc_now(),
    }

    try:
        fd = os.open(
            FINALIZER_LOCK_PATH,
            os.O_CREAT | os.O_EXCL | os.O_WRONLY,
            0o600,
        )
    except FileExistsError:
        existing = load_json(FINALIZER_LOCK_PATH, None)
        raise RuntimeError(
            "Another finalizer lock already exists. Do not run two wave finalizers "
            "concurrently.\n"
            f"Lock: {FINALIZER_LOCK_PATH}\n"
            f"Metadata: {existing}"
        )

    try:
        os.write(
            fd,
            (json.dumps(metadata, indent=2) + "\n").encode("utf-8"),
        )
    finally:
        os.close(fd)

    return metadata


def release_finalizer_lock():
    try:
        FINALIZER_LOCK_PATH.unlink()
    except FileNotFoundError:
        pass


def load_or_initialize_journal() -> Dict[str, Any]:
    existing = load_json(FINALIZATION_JOURNAL, None)

    if existing is not None:
        if existing.get("build_id") != BUILD_ID or existing.get("wave_id") != WAVE_ID:
            raise RuntimeError(
                "Existing finalization journal belongs to a different build/wave."
            )
        return existing

    payload = {
        "schema_version": "1.0",
        "artifact_type": "concurrent_preload_wave_finalization_journal",
        "build_id": BUILD_ID,
        "wave_id": WAVE_ID,
        "expected_states": EXPECTED_STATES,
        "status": "started",
        "started_at": utc_now(),
        "updated_at": utc_now(),
        "crop_merge": {},
        "snapshot": {},
    }
    write_json_atomic(FINALIZATION_JOURNAL, payload)
    return payload


def update_journal(journal: Dict[str, Any], **updates):
    journal.update(updates)
    journal["updated_at"] = utc_now()
    write_json_atomic(FINALIZATION_JOURNAL, journal)


def build_merged_crop_payload(current_global: Dict[str, Any]) -> Dict[str, Any]:
    merged = json.loads(json.dumps(current_global))

    for state, result in STATE_RESULTS.items():
        crop = result["crop_payload"]
        state_key = normalize_name(crop["state_key"])
        merged[state_key] = crop["crop_occurrences"]

    return merged


def apply_or_recover_crop_merge(journal: Dict[str, Any]) -> Dict[str, Any]:
    current = load_json(GLOBAL_CROP_JSON)
    if not isinstance(current, dict):
        raise RuntimeError("Global crop JSON is not a top-level object.")

    current_sha = file_sha256(GLOBAL_CROP_JSON)
    crop_journal = journal.setdefault("crop_merge", {})
    recorded_after = crop_journal.get("after_sha256")

    if recorded_after and current_sha == recorded_after:
        print("Crop merge already recorded and current global file matches it.")
        return crop_journal

    # Normal first-time merge.
    if current_sha == WAVE_SEED_SHA:
        backup_name = (
            f"crop_occurrences.before_{BUILD_ID}_{WAVE_ID}_"
            f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json"
        )
        backup_path = CROP_BACKUP_DIR / backup_name
        backup_path.write_bytes(GLOBAL_CROP_JSON.read_bytes())

        merged = build_merged_crop_payload(current)

        journal["status"] = "crop_merging"
        crop_journal.update({
            "before_sha256": current_sha,
            "backup_path": str(backup_path),
            "started_at": utc_now(),
        })
        update_journal(journal)

        write_json_atomic(GLOBAL_CROP_JSON, merged)

        after_sha = file_sha256(GLOBAL_CROP_JSON)
        crop_journal.update({
            "after_sha256": after_sha,
            "completed_at": utc_now(),
            "states_merged": EXPECTED_STATES,
        })
        journal["status"] = "crop_merged"
        update_journal(journal)

        print("✅ Global crop JSON merged atomically.")
        print("Backup:", backup_path)
        print("Before SHA:", current_sha)
        print("After SHA:", after_sha)
        return crop_journal

    # Crash-recovery window: os.replace may have succeeded before journal update.
    sections_match = True
    for state, result in STATE_RESULTS.items():
        crop = result["crop_payload"]
        key = normalize_name(crop["state_key"])
        if current.get(key) != crop["crop_occurrences"]:
            sections_match = False
            break

    if journal.get("status") in {
        "started",
        "crop_merging",
        "crop_merged",
        "snapshotting",
    } and sections_match:
        crop_journal.update({
            "after_sha256": current_sha,
            "completed_at": crop_journal.get("completed_at") or utc_now(),
            "states_merged": EXPECTED_STATES,
            "recovered_after_unrecorded_atomic_replace": True,
        })
        journal["status"] = "crop_merged"
        update_journal(journal)
        print("✅ Recovered previously applied crop merge.")
        return crop_journal

    raise RuntimeError(
        "Global crop JSON is neither the expected wave seed nor a safely "
        "recoverable already-applied merge."
    )


def list_qdrant_snapshots() -> List[Dict[str, Any]]:
    r = qdrant_get(
        f"/collections/{QDRANT_COLLECTION}/snapshots",
        timeout=120,
    )
    data = r.json()
    result = data.get("result", [])
    return result if isinstance(result, list) else []


def create_qdrant_snapshot() -> Dict[str, Any]:
    r = qdrant_post(
        f"/collections/{QDRANT_COLLECTION}/snapshots",
        payload=None,
        timeout=None,
    )
    data = r.json()

    if not isinstance(data, dict) or data.get("status") != "ok":
        raise RuntimeError(f"Snapshot creation failed: {data}")

    snapshot = data.get("result") or {}
    name = snapshot.get("name")
    if not name:
        raise RuntimeError(f"Qdrant did not return a snapshot name: {data}")

    return snapshot


def download_qdrant_snapshot(snapshot_name: str) -> Dict[str, Any]:
    destination = SNAPSHOT_OUTPUT_DIR / snapshot_name

    if destination.exists():
        return {
            "downloaded": True,
            "path": str(destination),
            "sha256": file_sha256(destination),
            "size_bytes": destination.stat().st_size,
            "reused_existing_file": True,
        }

    with qdrant_get(
        f"/collections/{QDRANT_COLLECTION}/snapshots/{snapshot_name}",
        stream=True,
        timeout=None,
    ) as response:
        tmp = destination.with_name(destination.name + ".tmp")
        with open(tmp, "wb") as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
        os.replace(tmp, destination)

    return {
        "downloaded": True,
        "path": str(destination),
        "sha256": file_sha256(destination),
        "size_bytes": destination.stat().st_size,
        "reused_existing_file": False,
    }


def ensure_snapshot(journal: Dict[str, Any]) -> Dict[str, Any]:
    snapshot_journal = journal.setdefault("snapshot", {})

    # Already recorded.
    if snapshot_journal.get("server_snapshot_name"):
        snapshot_name = snapshot_journal["server_snapshot_name"]

        if DOWNLOAD_SNAPSHOT:
            download_info = snapshot_journal.get("download")
            if not download_info or not Path(download_info.get("path", "")).exists():
                snapshot_journal["download"] = download_qdrant_snapshot(snapshot_name)
                update_journal(journal)

        return snapshot_journal

    # Mark intent before asking Qdrant. This gives us a recovery signal if the
    # kernel dies after Qdrant creates the snapshot but before we record its name.
    snapshot_journal["request_started_at"] = utc_now()
    snapshot_journal["qdrant_point_count"] = qdrant_exact_count()
    journal["status"] = "snapshotting"
    update_journal(journal)

    snapshot = create_qdrant_snapshot()
    snapshot_name = snapshot["name"]

    snapshot_journal.update({
        "server_snapshot_name": snapshot_name,
        "server_snapshot": snapshot,
        "created_at": utc_now(),
        "qdrant_point_count": qdrant_exact_count(),
    })
    update_journal(journal)

    if DOWNLOAD_SNAPSHOT:
        snapshot_journal["download"] = download_qdrant_snapshot(snapshot_name)
        update_journal(journal)

    print("✅ Qdrant cumulative snapshot created:", snapshot_name)
    return snapshot_journal


def build_wave_manifest(
    journal: Dict[str, Any],
    qdrant_count_before_snapshot: int,
) -> Dict[str, Any]:
    state_entries = {}

    for state, result in STATE_RESULTS.items():
        manifest = result["manifest"]
        state_entries[state] = {
            "state_manifest": str(result["manifest_path"]),
            "state_manifest_sha256": result["manifest_sha256"],
            "state_crop_file": str(result["crop_path"]),
            "state_crop_sha256": result["crop_sha256"],
            "completed_at": manifest.get("completed_at"),
            "state_statistics": manifest.get("state_statistics"),
            "metadata_quality": manifest.get("metadata_quality"),
        }

    global_crop_sha = file_sha256(GLOBAL_CROP_JSON)

    return {
        "schema_version": "1.0",
        "artifact_type": "concurrent_preload_wave_manifest",
        "status": "complete",
        "build_id": BUILD_ID,
        "wave_id": WAVE_ID,
        "expected_states": EXPECTED_STATES,
        "state_count": len(EXPECTED_STATES),
        "states": state_entries,
        "contracts": CROSS_STATE_CONTRACTS,
        "wave_statistics": WAVE_STATS,
        "global_crop": {
            "path": str(GLOBAL_CROP_JSON),
            "sha256": global_crop_sha,
            "seed_sha256": WAVE_SEED_SHA,
            "merge": journal.get("crop_merge"),
        },
        "qdrant": {
            "url": QDRANT_URL,
            "collection": QDRANT_COLLECTION,
            "point_count": qdrant_exact_count(),
            "point_count_before_snapshot": qdrant_count_before_snapshot,
            "sample_payload_contract_warnings": sample_contract_issues,
            "snapshot": journal.get("snapshot"),
        },
        "coordinator": {
            "url": COORDINATOR_URL,
        },
        "finalizer": {
            "hostname": socket.gethostname(),
            "pid": os.getpid(),
        },
        "completed_at": utc_now(),
    }


print("Finalization implementation ready.")


## 10. Run finalization

Before setting `RUN_FINALIZATION=True`, confirm:

- every expected state is complete;
- no state worker in this wave is still writing;
- the service node is stable;
- `GLOBAL_CROP_JSON` points to the maintained global crop file;
- Qdrant uses the intended persistent storage path.

The wave manifest is written **last**.


In [ ]:
def finalize_wave():
    if WAVE_MANIFEST_PATH.exists():
        existing = load_json(WAVE_MANIFEST_PATH)
        if (
            isinstance(existing, dict)
            and existing.get("status") == "complete"
            and existing.get("build_id") == BUILD_ID
            and existing.get("wave_id") == WAVE_ID
            and existing.get("expected_states") == EXPECTED_STATES
        ):
            print("Wave already finalized. Returning existing wave manifest.")
            return existing

        raise RuntimeError(
            f"Wave manifest already exists but does not match this configuration: "
            f"{WAVE_MANIFEST_PATH}"
        )

    if not RUN_FINALIZATION:
        raise RuntimeError(
            "RUN_FINALIZATION is False. Validation has succeeded, but no global "
            "mutation was performed. Set RUN_FINALIZATION=True only when ready."
        )

    lock_info = acquire_finalizer_lock()
    print("Finalizer lock acquired:", lock_info)

    try:
        # Re-run service + state validation immediately before mutating globals.
        service_preflight()

        refreshed_states = {
            state: load_and_validate_state(state)
            for state in EXPECTED_STATES
        }

        if set(refreshed_states) != set(STATE_RESULTS):
            raise RuntimeError("Expected-state set changed during finalization.")

        journal = load_or_initialize_journal()

        if journal.get("status") == "complete" and WAVE_MANIFEST_PATH.exists():
            return load_json(WAVE_MANIFEST_PATH)

        # Global crop merge.
        apply_or_recover_crop_merge(journal)

        # Qdrant validation immediately before snapshot.
        qdrant_count = qdrant_exact_count()
        if qdrant_count <= 0:
            raise RuntimeError("Qdrant has zero points before snapshot.")

        # Snapshot.
        ensure_snapshot(journal)

        # Final wave manifest is the commit marker and must be written last.
        wave_manifest = build_wave_manifest(journal, qdrant_count)
        write_json_atomic(WAVE_MANIFEST_PATH, wave_manifest)

        # Mark journal complete only after the manifest exists.
        journal["status"] = "complete"
        journal["wave_manifest_path"] = str(WAVE_MANIFEST_PATH)
        journal["wave_manifest_sha256"] = file_sha256(WAVE_MANIFEST_PATH)
        journal["completed_at"] = utc_now()
        update_journal(journal)

        print("\n✅ WAVE FINALIZATION COMPLETE")
        print("Wave manifest:", WAVE_MANIFEST_PATH)
        print("Wave manifest SHA256:", file_sha256(WAVE_MANIFEST_PATH))
        print("Global crop JSON:", GLOBAL_CROP_JSON)
        print("Global crop SHA256:", file_sha256(GLOBAL_CROP_JSON))
        print(
            "Qdrant snapshot:",
            (journal.get("snapshot") or {}).get("server_snapshot_name"),
        )

        return wave_manifest

    except Exception as exc:
        try:
            journal = load_json(FINALIZATION_JOURNAL, {}) or {}
            if isinstance(journal, dict):
                journal["last_error"] = {
                    "type": type(exc).__name__,
                    "message": str(exc),
                    "time": utc_now(),
                }
                journal["updated_at"] = utc_now()
                write_json_atomic(FINALIZATION_JOURNAL, journal)
        except Exception:
            pass
        raise

    finally:
        release_finalizer_lock()


if RUN_FINALIZATION:
    WAVE_MANIFEST = finalize_wave()
else:
    print(
        "✅ Validation/preview complete.\n"
        "No global mutation performed because RUN_FINALIZATION=False.\n"
        "When ready, set RUN_FINALIZATION=True and rerun this cell."
    )


## 11. Inspect finalization artifacts


In [ ]:
def show_finalization_status():
    print("Build:", BUILD_ID)
    print("Wave:", WAVE_ID)
    print("Expected states:", EXPECTED_STATES)
    print()

    if FINALIZATION_JOURNAL.exists():
        print("=== Finalization journal ===")
        print(FINALIZATION_JOURNAL.read_text(encoding="utf-8"))
    else:
        print("No finalization journal yet.")

    print()

    if WAVE_MANIFEST_PATH.exists():
        print("=== Wave manifest ===")
        print(WAVE_MANIFEST_PATH.read_text(encoding="utf-8"))
    else:
        print("No completed wave manifest yet.")


show_finalization_status()


## 12. Recovery notes

### A state is incomplete

Do not finalize. Resume that worker state until both:

- its `state_manifest.json` says `complete`; and
- coordinator `/state/status` says `complete`.

### Finalizer crashes before crop merge

Rerun the notebook. The journal remains recoverable and the global crop file remains at the wave seed.

### Finalizer crashes after atomic crop replace

Rerun the notebook. The recovery logic accepts the already-merged crop file only when the expected wave state sections exactly match their state crop artifacts.

### Finalizer crashes during snapshot creation

Rerunning may create another Qdrant server snapshot if the previous snapshot was created but its name was never recorded. This is safe for database correctness; it may leave an extra snapshot on the server. The wave manifest references only the snapshot recorded by the successful finalization.

### Snapshot restore

Restore is intentionally **not implemented here**. A restore is a build-level administrative operation and must only be performed while all workers are stopped.

### Global crop JSON

The maintained global `crop_occurrences.json` persists across waves. Each new wave starts from the version finalized by the previous wave.
